# Ranked gene loadings for hard-to-distinguish cell-type pairs

For each biologically related pair in `RELATED_GROUPS` (the same six groups as
`04_hard_cell_types.ipynb`), this notebook takes the LV assigned to each member and
ranks every gene by its CLAMPfull loading, then overlays the two competing marker
sets. It asks a sharper question than the marker dot grid: not *is this LV enriched
for these markers*, but *does this LV rank one cell type's markers above the other's*.

Two design points that matter for interpreting the output.

**Shared markers are removed from both sets and reported separately.** Only markers
exclusive to one side can discriminate a pair. Genes annotated to both are kept in the
output with `grp = "shared"` so they can be drawn in their own colour rather than
silently dropped into the background.

**Azimuth is the canonical marker resource.** Every displayed and reported pair uses
the dataset-matched Azimuth reference atlas, consistent with the pseudobulk ORA.
CellMarker and Allen remain in the resource sweep as sensitivity analyses only; they
never select the primary result. This avoids choosing a database after observing which
one best separates a pair.

No model is fitted or altered here.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
suppressPackageStartupMessages({
  library(here); library(data.table); library(readxl)
})

## Settings and inputs

In [ ]:
OUT_DIR <- here(snakemake@params[["out_dir"]])
dir.create(OUT_DIR, recursive = TRUE, showWarnings = FALSE)

# LV assignments come from the canonical table, never from a fresh argmax over the
# correlation matrix. top_lvs_per_celltype.csv resolves ties so that no LV is
# assigned to two cell types in the same dataset; recomputing the argmax here would
# reintroduce collisions (e.g. Brain_Xiong2023 LV5 is the best LV for astrocyte AND
# the second-best for OPC).
top_lvs  <- fread(snakemake@input[["top"]])
lv_corr  <- fread(snakemake@input[["corr"]])
stopifnot(top_lvs[, .N, by = .(dataset, LV)][N > 1, .N] == 0)

TOP_GENE_PCT <- 0.01   # matches get_top_genes() elsewhere in the project
N_LABEL      <- 8      # candidate gene labels per side; panels may show fewer
ALLEN_CONSENSUS <- 0.5

ct_labels_df <- read.csv(here("data", "pseudobulk", "cell_type_labels.csv"),
                         stringsAsFactors = FALSE)
CT_LABELS <- setNames(ct_labels_df$label, ct_labels_df$cell_type)
ct_label <- function(x) ifelse(x %in% names(CT_LABELS), CT_LABELS[x], x)

## Marker resources

Three databases, each read into a named list of gene vectors.

In [ ]:
cm <- as.data.table(read_excel(snakemake@input[["cell_marker_file"]], sheet = "human"))
cm <- cm[!is.na(Symbol) & !is.na(cell_name) & species == "Human"]
CELLMARKER <- lapply(split(cm$Symbol, cm$cell_name), unique)

read_tsv_sets <- function(path) {
  sp <- strsplit(readLines(path, warn = FALSE), "\t", fixed = TRUE)
  setNames(lapply(sp, function(x) { g <- x[-c(1, 2)]; unique(g[nzchar(g)]) }),
           vapply(sp, `[`, character(1), 1))
}
AZIMUTH <- read_tsv_sets(snakemake@input[["azimuth_file"]])
ALLEN   <- read_tsv_sets(snakemake@input[["allen_brain_gmt_file"]])

# Allen ships one set per subtype cluster, split into "up" and "down". Unioning a
# cell type's clusters yields "genes seen anywhere in that lineage" and destroys the
# discrimination the clusters were built to provide. The consensus -- present in at
# least ALLEN_CONSENSUS of the type's human "up" clusters -- restores it: astrocyte
# vs microglia goes from 120/39 genes with 1.2% overlap to 52/41 with none.
allen_consensus <- function(pattern, frac = ALLEN_CONSENSUS) {
  s <- ALLEN[grepl(pattern, names(ALLEN)) & grepl(" up$", names(ALLEN))]
  if (!length(s)) stop("no Allen clusters match: ", pattern)
  tb <- table(unlist(s))
  g <- names(tb)[tb >= frac * length(s)]
  if (!length(g))
    stop(sprintf("Allen consensus empty at frac=%.2f for '%s' (%d clusters share no genes)",
                 frac, pattern, length(s)))
  g
}

get_set <- function(src, nm) {
  if (src == "allen") return(allen_consensus(nm))
  s <- switch(src, cellmarker = CELLMARKER, azimuth = AZIMUTH,
              stop("unknown marker source: ", src))
  if (!nm %in% names(s)) stop("set not found in ", src, ": ", nm)
  s[[nm]]
}

cat(sprintf("CellMarker %d sets | Azimuth %d | Allen %d\n",
            length(CELLMARKER), length(AZIMUTH), length(ALLEN)))

## The six pairs

Same groups as `04_hard_cell_types.ipynb`. All primary pairs use the corresponding
Azimuth atlas; `shown` marks the four pairs the published panels display.

In [ ]:
PAIRS <- list(
  list(id = "01_B_vs_T", shown = TRUE, src = "azimuth",
       A = "PBMC-L1-B Cell", B = "PBMC-L1-CD4+ T Cell",
       Alab = "B cell", Blab = "T cell",
       members = data.table(side = c("A", "B"),
                            ds = c("PBMC_Perez2022", "PBMC_Perez2022"),
                            ct = c("B_cell", "T_cell"))),
  list(id = "02_CD4_vs_CD8", shown = TRUE, src = "azimuth",
       A = "PBMC-L1-CD4+ T Cell", B = "PBMC-L1-CD8+ T Cell",
       Alab = "CD4+ T cell", Blab = "CD8+ T cell",
       members = data.table(side = c("A", "B"),
                            ds = c("PBMC_1k1k", "PBMC_1k1k"),
                            ct = c("CD4_T", "CD8_T"))),
  list(id = "03_OPC_vs_Oli", shown = FALSE, src = "azimuth",
       A = "Motor Cortex-subclass-oligodendrocyte Precursor Cell",
       B = "Motor Cortex-subclass-oligodendrocyte",
       Alab = "OPC", Blab = "Oligodendrocyte",
       members = data.table(side = c("A", "B"),
                            ds = c("Brain_Xiong2023", "Brain_Xiong2023"),
                            ct = c("Opc", "Oli"))),
  list(id = "04_Ast_vs_Mic", shown = TRUE, src = "azimuth",
       A = "Motor Cortex-subclass-Astrocyte",
       B = "Motor Cortex-subclass-microglia / Perivascular Macrophage",
       Alab = "Astrocyte", Blab = "Microglia",
       members = data.table(side = c("A", "B"),
                            ds = c("Brain_Mathys2023", "Brain_Mathys2023"),
                            ct = c("Ast", "Mic"))),
  list(id = "05_Inh_vs_Exc", shown = FALSE, src = "azimuth",
       A = "Motor Cortex-class-GABAergic Neuron",
       B = "Motor Cortex-class-Glutamatergic Neuron",
       Alab = "Inhibitory neuron", Blab = "Excitatory neuron",
       members = data.table(side = c("A", "B"),
                            ds = c("Brain_Mathys2023", "Brain_Xiong2023"),
                            ct = c("Inh", "Exc"))),
  list(id = "06_Atrial_vs_Ventricular", shown = TRUE, src = "azimuth",
       A = "Heart-L2-Atrial Cardiomyocyte", B = "Heart-L2-Ventricular Cardiomyocyte",
       Alab = "Atrial CM", Blab = "Ventricular CM",
       members = data.table(side = c("A", "B"),
                            ds = c("Heart_Datar2026", "Heart_Datar2026"),
                            ct = c("AtrialCM", "VentricularCM")))
)

# Weak-LV threshold. OPC is the pair that trips it: its assigned LV11 is clean
# (Opc 0.484, next-best Oli 0.034) but no LV in either brain model tracks OPC above
# r = 0.49. Recorded as a flag rather than an error so the pair stays documented.
MIN_R_TRUTH <- 0.60

## Helpers

`auc_membership` is the same statistic as the `AUC` column of CLAMPfull's
`summary.csv`, so these numbers are directly comparable to it.

In [ ]:
auc_membership <- function(loading, in_set) {
  npos <- sum(in_set); nneg <- sum(!in_set)
  if (npos == 0 || nneg == 0) return(NA_real_)
  r <- rank(loading)
  (sum(r[in_set]) - npos * (npos + 1) / 2) / (npos * nneg)
}

Z_cache <- new.env()
get_Z <- function(ds) {
  if (is.null(Z_cache[[ds]]))
    Z_cache[[ds]] <- fread(here(sprintf(
      "output/01_model_building/00_pseudobulk/%s/models/CLAMPfull/Z.csv", ds)))
  Z_cache[[ds]]
}

lv_of <- function(ds, ct) {
  v <- top_lvs[dataset == ds & cell_type == ct, LV]
  if (!length(v)) stop("no assigned LV for ", ds, " / ", ct)
  v[1]
}
r_of <- function(ds, ct, lv) {
  v <- lv_corr[dataset == ds & cell_type == ct & LV == lv, cor]
  if (length(v)) v[1] else NA_real_
}

## Score every panel

For each pair, each of the two LVs gets its own panel. Positives are the exclusive
markers of one side; the Wilcoxon compares the two exclusive sets' loadings over
**all** model genes, zeros included.

Restricting the test to non-zero genes, as an earlier version did, conditions on the
outcome: a competing marker set driven entirely to zero by CLAMP is the strongest
evidence of specificity available, and dropping those genes threw it away and returned
`NA` on 5 of 12 panels.

In [ ]:
panel_rows <- list(); stat_rows <- list()

for (P in PAIRS) {
  setA_raw <- get_set(P$src, P$A)
  setB_raw <- get_set(P$src, P$B)

  for (i in seq_len(nrow(P$members))) {
    m    <- P$members[i]
    lv   <- lv_of(m$ds, m$ct)
    ct_o <- P$members[side != m$side, ct][1]

    Z  <- get_Z(m$ds)
    if (!lv %in% names(Z)) stop(m$ds, " has no ", lv)
    dt <- data.table(gene = Z[[1]], loading = Z[[lv]])[!is.na(gene) & nzchar(gene)]

    A_full <- intersect(setA_raw, dt$gene)
    B_full <- intersect(setB_raw, dt$gene)
    shared <- intersect(A_full, B_full)
    A <- setdiff(A_full, shared); B <- setdiff(B_full, shared)

    aucA <- auc_membership(dt$loading, dt$gene %in% A)
    aucB <- auc_membership(dt$loading, dt$gene %in% B)
    lA <- dt[gene %in% A, loading]; lB <- dt[gene %in% B, loading]
    wt <- if (length(lA) > 2 && length(lB) > 2)
      suppressWarnings(wilcox.test(lA, lB)) else NULL

    # Non-zero genes only: what the panel actually draws.
    nz <- dt[loading != 0][order(-loading)]
    nz[, rank := .I]
    nz[, grp := fifelse(gene %in% A, "A",
                fifelse(gene %in% B, "B",
                fifelse(gene %in% shared, "shared", "other")))]
    # nz is already ordered by descending loading, so rowid() counts within each
    # group in rank order and the first N_LABEL of each side are the candidates.
    nz[, is_label := grp %in% c("A", "B") & rowid(grp) <= N_LABEL]

    r_own   <- r_of(m$ds, m$ct, lv)
    r_other <- r_of(m$ds, ct_o, lv)
    own     <- if (m$side == "A") aucA else aucB
    other   <- if (m$side == "A") aucB else aucA

    panel_rows[[length(panel_rows) + 1L]] <- data.table(
      pair_id = P$id, shown = P$shown, panel_side = m$side,
      dataset = m$ds, cell_type = m$ct, LV = lv, marker_source = "Azimuth 2023",
      set_A_label = P$Alab, set_B_label = P$Blab,
      gene = nz$gene, rank = nz$rank, loading = nz$loading,
      grp = nz$grp, is_label = nz$is_label)

    stat_rows[[length(stat_rows) + 1L]] <- data.table(
      pair_id = P$id, shown = P$shown, panel_side = m$side,
      dataset = m$ds, cell_type = m$ct, LV = lv,
      marker_source = "Azimuth 2023", set_A = P$A, set_B = P$B,
      set_A_label = P$Alab, set_B_label = P$Blab,
      r_truth = r_own, r_competitor = r_other,
      weak_lv = is.na(r_own) || r_own < MIN_R_TRUTH,
      n_A_raw = length(setA_raw), n_B_raw = length(setB_raw),
      n_A_excl = length(A), n_B_excl = length(B), n_shared = length(shared),
      n_genes_nonzero = nrow(nz),
      frac_zero_A = if (length(lA)) mean(lA == 0) else NA_real_,
      frac_zero_B = if (length(lB)) mean(lB == 0) else NA_real_,
      auc_A = aucA, auc_B = aucB,
      auc_own = own, auc_other = other, auc_gap = own - other,
      wilcox_p = if (!is.null(wt)) wt$p.value else NA_real_)
  }
}

panel_ready <- rbindlist(panel_rows)
stats <- rbindlist(stat_rows)
stats[, correct := !is.na(auc_gap) & auc_gap > 0]
stats[, significant := !is.na(wilcox_p) & wilcox_p < 0.05]
cat(sprintf("%d panels, %d plotted gene rows\n", nrow(stats), nrow(panel_ready)))

## Results

In [ ]:
cat("=== PER-PANEL SEPARATION ===\n")
print(as.data.frame(stats[, .(pair_id, LV, cell_type,
  src = marker_source, nA = n_A_excl, nB = n_B_excl, sh = n_shared,
  r_truth = round(r_truth, 3), r_comp = round(r_competitor, 3),
  auc_own = round(auc_own, 3), auc_other = round(auc_other, 3),
  gap = round(auc_gap, 3), p = signif(wilcox_p, 2),
  correct, significant, weak_lv, shown)]), row.names = FALSE)

cat(sprintf("\nAll pairs      : %d/%d panels correct, %d significant\n",
            sum(stats$correct), nrow(stats), sum(stats$significant)))
cat(sprintf("Shown pairs only: %d/%d panels correct, %d significant\n",
            stats[shown == TRUE, sum(correct)], stats[shown == TRUE, .N],
            stats[shown == TRUE, sum(significant)]))

wk <- stats[weak_lv == TRUE]
if (nrow(wk)) {
  cat(sprintf("\nWeak LVs (r_truth < %.2f), kept and flagged, not shown in panels:\n",
              MIN_R_TRUTH))
  print(as.data.frame(wk[, .(pair_id, LV, cell_type,
                             r_truth = round(r_truth, 3),
                             r_competitor = round(r_competitor, 3))]),
        row.names = FALSE)
}

## Marker-resource sweep

The same statistic against every resource that carries both sides of a pair. This is
what selects the per-pair resource above, and it is the evidence that the limiting
instrument on the hard pairs is the marker database rather than the model.

In [ ]:
SWEEP <- list(
  list(id = "01_B_vs_T", res = list(
    list(tag = "CellMarker", src = "cellmarker", A = "B cell", B = "T cell"),
    list(tag = "Azimuth PBMC-L1", src = "azimuth",
         A = "PBMC-L1-B Cell", B = "PBMC-L1-CD4+ T Cell"))),
  list(id = "02_CD4_vs_CD8", res = list(
    list(tag = "CellMarker", src = "cellmarker", A = "CD4+ T cell", B = "CD8+ T cell"),
    list(tag = "Azimuth PBMC-L1", src = "azimuth",
         A = "PBMC-L1-CD4+ T Cell", B = "PBMC-L1-CD8+ T Cell"))),
  list(id = "03_OPC_vs_Oli", res = list(
    list(tag = "CellMarker", src = "cellmarker",
         A = "Oligodendrocyte precursor cell", B = "Oligodendrocyte"),
    list(tag = "Azimuth MotorCortex", src = "azimuth",
         A = "Motor Cortex-subclass-oligodendrocyte Precursor Cell",
         B = "Motor Cortex-subclass-oligodendrocyte"),
    list(tag = "Allen consensus", src = "allen", A = "^Human OPC", B = "^Human Oligo"))),
  list(id = "04_Ast_vs_Mic", res = list(
    list(tag = "CellMarker", src = "cellmarker", A = "Astrocyte", B = "Microglial cell"),
    list(tag = "Azimuth MotorCortex", src = "azimuth",
         A = "Motor Cortex-subclass-Astrocyte",
         B = "Motor Cortex-subclass-microglia / Perivascular Macrophage"),
    list(tag = "Allen consensus", src = "allen", A = "^Human Astro", B = "^Human Micro"))),
  list(id = "05_Inh_vs_Exc", res = list(
    list(tag = "CellMarker", src = "cellmarker",
         A = "Inhibitory neuron", B = "Excitatory neuron"),
    list(tag = "Azimuth MotorCortex", src = "azimuth",
         A = "Motor Cortex-class-GABAergic Neuron",
         B = "Motor Cortex-class-Glutamatergic Neuron"))),
  list(id = "06_Atrial_vs_Ventricular", res = list(
    list(tag = "Azimuth Heart-L2", src = "azimuth",
         A = "Heart-L2-Atrial Cardiomyocyte", B = "Heart-L2-Ventricular Cardiomyocyte")))
)

PAIR_BY_ID <- setNames(PAIRS, vapply(PAIRS, `[[`, character(1), "id"))

sweep_rows <- list()
for (S in SWEEP) {
  P <- PAIR_BY_ID[[S$id]]
  for (R in S$res) {
    # Allen has no consensus set for inhibitory neurons: its 60 human Inh clusters
    # share no genes at 50%. Catch it so the sweep reports the gap instead of dying.
    sets <- tryCatch(list(A = get_set(R$src, R$A), B = get_set(R$src, R$B)),
                     error = function(e) { message(S$id, " / ", R$tag, ": ", conditionMessage(e)); NULL })
    if (is.null(sets)) next
    for (i in seq_len(nrow(P$members))) {
      m  <- P$members[i]
      lv <- lv_of(m$ds, m$ct)
      Z  <- get_Z(m$ds)
      dt <- data.table(gene = Z[[1]], loading = Z[[lv]])[!is.na(gene) & nzchar(gene)]
      A_full <- intersect(sets$A, dt$gene); B_full <- intersect(sets$B, dt$gene)
      sh <- intersect(A_full, B_full)
      A <- setdiff(A_full, sh); B <- setdiff(B_full, sh)
      aucA <- auc_membership(dt$loading, dt$gene %in% A)
      aucB <- auc_membership(dt$loading, dt$gene %in% B)
      lA <- dt[gene %in% A, loading]; lB <- dt[gene %in% B, loading]
      wt <- if (length(lA) > 2 && length(lB) > 2)
        suppressWarnings(wilcox.test(lA, lB)) else NULL
      own   <- if (m$side == "A") aucA else aucB
      other <- if (m$side == "A") aucB else aucA
      sweep_rows[[length(sweep_rows) + 1L]] <- data.table(
        pair_id = S$id, resource = R$tag, LV = lv, cell_type = m$ct,
        n_A_excl = length(A), n_B_excl = length(B), n_shared = length(sh),
        pct_shared = round(100 * length(sh) / max(1L, length(union(A_full, B_full))), 1),
        auc_own = own, auc_other = other, auc_gap = own - other,
        wilcox_p = if (!is.null(wt)) wt$p.value else NA_real_)
    }
  }
}
sweep <- rbindlist(sweep_rows)
sweep[, significant := !is.na(wilcox_p) & wilcox_p < 0.05]

cat("\n=== MARKER-RESOURCE SWEEP ===\n")
print(as.data.frame(sweep[, .(mean_gap = round(mean(auc_gap, na.rm = TRUE), 3),
                              n_sig = sum(significant), n = .N),
                          by = .(pair_id, resource)][order(pair_id, -mean_gap)]),
      row.names = FALSE)

## Figure preview

One panel per latent variable, two per pair. Genes are ranked by loading; the two
competing marker sets are overlaid, with genes annotated to both drawn in plum rather
than dropped into the background. The subtitle reads as a verdict: the winning AUROC
first, `>` when the difference is significant and `~` when it is not.

In [ ]:
suppressPackageStartupMessages({
  library(ggplot2); library(ggrepel); library(patchwork)
})

BLUE <- "#2166AC"; ORANGE <- "#E08214"; GREY <- "#BFBFBF"
# Shared markers get their own hue: at print size a darker grey is indistinguishable
# from the background, so genes annotated to BOTH cell types read as unannotated.
SHARED <- "#762A83"

make_pair_panel <- function(pid) {
  pr <- panel_ready[pair_id == pid]
  st <- stats[pair_id == pid]
  Alab <- st$set_A_label[1]; Blab <- st$set_B_label[1]
  cols <- setNames(c(BLUE, ORANGE, SHARED), c(Alab, Blab, "shared marker"))

  panels <- lapply(c("A", "B"), function(side) {
    d <- pr[panel_side == side]
    s <- st[panel_side == side]
    d[, grp_lab := factor(fifelse(grp == "A", Alab,
                          fifelse(grp == "B", Blab,
                          fifelse(grp == "shared", "shared marker", "other"))),
                          levels = c(Alab, Blab, "shared marker", "other"))]

    hi_first <- s$auc_A >= s$auc_B
    l1 <- if (hi_first) Alab else Blab; a1 <- if (hi_first) s$auc_A else s$auc_B
    l2 <- if (hi_first) Blab else Alab; a2 <- if (hi_first) s$auc_B else s$auc_A
    sep <- if (!is.na(s$wilcox_p) && s$wilcox_p < 0.05) ">" else "~"
    pstr <- if (is.na(s$wilcox_p)) "no test" else
      paste0("P = ", format.pval(s$wilcox_p, digits = 2, eps = 1e-300),
             if (s$wilcox_p >= 0.05) " (n.s.)" else "")
    sub <- sprintf("AUROC %s %.3f %s %s %.3f | %s", l1, a1, sep, l2, a2, pstr)

    ggplot(d, aes(rank, loading)) +
      geom_point(data = d[grp_lab == "other"], colour = GREY, size = 1.0) +
      geom_point(data = d[grp_lab == "shared marker"], aes(colour = grp_lab), size = 1.9) +
      geom_point(data = d[grp %in% c("A", "B")], aes(colour = grp_lab), size = 1.9) +
      geom_text_repel(data = d[is_label == TRUE], aes(label = gene, colour = grp_lab),
                      size = 2.6, fontface = "italic", direction = "y", hjust = 0,
                      seed = 42, max.overlaps = Inf, min.segment.length = 0,
                      box.padding = 0.3, point.padding = 0.2, nudge_x = 8,
                      segment.colour = "grey60", segment.size = 0.25,
                      show.legend = FALSE) +
      scale_colour_manual(values = cols,
                          breaks = c(Alab, Blab, "shared marker"), drop = FALSE) +
      coord_cartesian(clip = "off") +
      scale_x_continuous(expand = expansion(mult = c(0.02, 0.18))) +
      labs(title = sprintf("%s %s - %s (r = %.2f)", s$dataset, s$LV, s$cell_type,
                           s$r_truth),
           subtitle = sub, x = "Genes (ranked by loading)", y = "Loadings") +
      theme_classic(base_size = 11) +
      theme(plot.title = element_text(face = "bold", hjust = 0.5, size = 11),
            plot.subtitle = element_text(hjust = 0.5, size = 7.5, colour = "grey30"),
            legend.position = "top", legend.title = element_blank(),
            legend.key.size = unit(0.35, "cm"),
            plot.background = element_rect(fill = "white", colour = NA),
            plot.margin = margin(5.5, 26, 5.5, 5.5))
  })
  panels[[1]] | panels[[2]]
}

for (pid in unique(panel_ready$pair_id)) {
  shown <- stats[pair_id == pid, shown][1]
  cat(sprintf("\n%s%s\n", pid, if (shown) "" else "   [not shown in the figure panels]"))
  options(repr.plot.width = 11.2, repr.plot.height = 4.6)
  print(make_pair_panel(pid))
}

## Save

In [ ]:
fwrite(panel_ready, snakemake@output[["panel_ready"]])
fwrite(stats,       snakemake@output[["stats"]])
fwrite(sweep,       snakemake@output[["sweep"]])

cat("\nWrote:\n")
for (k in c("panel_ready", "stats", "sweep"))
  cat("  ", basename(snakemake@output[[k]]), "\n", sep = "")